In [1]:
# Install the required Python libraries
# These libraries will be used for loading models and running benchmarks.
!pip install -q -U transformers accelerate bitsandbytes sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.2 MB/s eta 0:00:00


In [2]:
# Import required libraries

import torch
import time
import gc
import pandas as pd

from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from transformers import BitsAndBytesConfig


In [3]:
# Check GPU

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

# Start measuring GPU memory for this model
torch.cuda.reset_peak_memory_stats()


2.11.0+cu128
True
Tesla T4


## Helper Functions

I created two helper functions to avoid repeating the same code. This keeps the notebook simple and organized.

In [4]:
# Clear GPU memory so the next model can load properly.

def free_gpu(*objects):
    for obj in objects:
        try:
            del obj
        except NameError:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    print("GPU memory cleared.")


In [42]:
# Save the benchmark results for each model in a table.

def build_result_row(model_name, response_text, inference_time, tokens_generated,
                      gpu_memory_gb, context_length, quantized=False):

    tokens_per_sec = tokens_generated / inference_time if inference_time > 0 else 0
    latency_ms = (inference_time / tokens_generated) * 1000 if tokens_generated > 0 else 0

    # Colab Free itself is not billed per run, so this converts the runtime
    # into what it WOULD cost on a typical on-demand cloud T4 (~$0.35/hour)
    # just so the numbers are comparable in cost terms.
    T4_HOURLY_RATE_USD = 0.35
    cost_estimate = (inference_time / 3600) * T4_HOURLY_RATE_USD

    sample = response_text.strip().replace("\n", " ")
    if len(sample) > 100:
        sample = sample[:100] + "..."

    return {
        "Model": model_name,
        "Quantized (4-bit)": quantized,
        "Response Quality (sample)": sample,
        "Inference Time (s)": round(inference_time, 2),
        "Tokens/sec": round(tokens_per_sec, 2),
        "GPU Memory (GB)": round(gpu_memory_gb, 2),
        "Context Length": context_length,
        "Latency (ms/token)": round(latency_ms, 2),
        "Cost Estimate (USD)": round(cost_estimate, 6),
    }


## Model 1: Qwen2.5-0.5B-Instruct

In [6]:
# Load Qwen Model

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

print("✅ Model Loaded Successfully!")


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Model Loaded Successfully!


In [7]:
# Ask the model a simple question

prompt = "What is Artificial Intelligence?"

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)


Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and act like humans. It involves developing algorithms and models that can process information, learn from data, and make decisions based on that learning.

Key characteristics of AI include:

1. **Learning**: The ability to improve its performance over time through experience.
2. **Reasoning**: The capacity to understand and reason about the world around it.
3. **Perception**: The capability to perceive and interpret


In [8]:
# Measure Inference Time

prompt = "Explain Machine Learning in one sentence."

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

start_time = time.time()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False
    )

end_time = time.time()

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)
print(f"\nInference Time: {end_time - start_time:.2f} seconds")


Machine learning is a subset of artificial intelligence that enables computers to learn from data and improve their performance on specific tasks through trial and error, without being explicitly programmed.

Inference Time: 1.91 seconds


In [9]:
# Measure GPU Memory Usage

if torch.cuda.is_available():
    gpu_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)
    print(f"GPU Memory Used: {gpu_memory:.2f} GB")
else:
    gpu_memory = 0
    print("GPU is not available.")


GPU Memory Used: 0.93 GB


In [10]:
# Save the benchmark results for this model.

tokens_generated = outputs.shape[1] - inputs["input_ids"].shape[1]
context_length = getattr(model.config, "max_position_embeddings", "N/A")

qwen_result = build_result_row(
    model_name="Qwen2.5-0.5B-Instruct",
    response_text=response,
    inference_time=end_time - start_time,
    tokens_generated=tokens_generated,
    gpu_memory_gb=gpu_memory,
    context_length=context_length,
    quantized=False
)

pd.DataFrame([qwen_result])


,Model,Quantized (4-bit),Response Quality (sample),Inference Time (s),Tokens/sec,GPU Memory (GB),Context Length,Latency (ms/token),Cost Estimate (USD)
0,Qwen2.5-0.5B-Instruct,False,Machine learning is a subset of artificial int...,1.91,17.25,0.93,32768,57.98,0.000186


In [11]:
# Save Results as CSV

pd.DataFrame([qwen_result]).to_csv("qwen_benchmark.csv", index=False)

print("Benchmark results saved successfully!")


Benchmark results saved successfully!


In [12]:
# Download the benchmark results as a CSV file.

from google.colab import files

files.download("qwen_benchmark.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Building a Simple Transformer Decoder Block

In [13]:
# Import required modules

import torch
import torch.nn as nn


In [14]:
# Build a simple Transformer decoder block.
# A causal mask is used so each token only attends

class SimpleDecoderBlock(nn.Module):
    def __init__(self, embed_dim=128, num_heads=8):
        super().__init__()

        self.attention = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(embed_dim)

        self.feed_forward = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.ReLU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )

        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        seq_len = x.size(1)

        # True = "not allowed to attend to this position"
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=x.device),
            diagonal=1
        ).bool()

        attention_output, _ = self.attention(x, x, x, attn_mask=causal_mask)

        x = self.norm1(x + attention_output)

        ff_output = self.feed_forward(x)

        x = self.norm2(x + ff_output)

        return x


In [15]:
# Test the Decoder Block

decoder = SimpleDecoderBlock()

sample_input = torch.rand(2, 10, 128)

output = decoder(sample_input)

print("Input Shape :", sample_input.shape)
print("Output Shape:", output.shape)


Input Shape : torch.Size([2, 10, 128])
Output Shape: torch.Size([2, 10, 128])


## Quantization with BitsAndBytes

In [16]:
import bitsandbytes as bnb

print("BitsAndBytes Version:", bnb.__version__)
print("✅ BitsAndBytes imported successfully!")


BitsAndBytes Version: 0.49.2
✅ BitsAndBytes imported successfully!


In [17]:
# Use 4-bit quantization to reduce GPU memory usage.

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("✅ Quantization Configuration Created Successfully!")


✅ Quantization Configuration Created Successfully!


## Model 2: Zephyr-7B-beta

This model is used to compare the performance of a Mistral-based language model. It works well on Google Colab and fits within the available GPU memory.

In [18]:
# Clear GPU memory before loading the next model.
free_gpu(model, tokenizer)


GPU memory cleared.


In [19]:
# Load Zephyr-7B-beta in 4-bit (Mistral-family substitute)

MODEL_NAME = "HuggingFaceH4/zephyr-7b-beta"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

print("✅ Zephyr-7B-beta Loaded Successfully (4-bit)!")


config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

✅ Zephyr-7B-beta Loaded Successfully (4-bit)!


In [20]:
# Benchmark Zephyr: same prompt, same method as every other model

prompt = "Explain Machine Learning in one sentence."

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

start_time = time.time()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False
    )

end_time = time.time()

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)
print(f"\nInference Time: {end_time - start_time:.2f} seconds")

if torch.cuda.is_available():
    gpu_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)
else:
    gpu_memory = 0

print(f"GPU Memory Used: {gpu_memory:.2f} GB")


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Machine learning is the field of computer science that enables computers to learn and make decisions based on data without being explicitly programmed, using algorithms and statistical models to continuously improve their performance over time.

Inference Time: 4.60 seconds
GPU Memory Used: 4.87 GB


In [21]:
# Save Zephyr's results row

tokens_generated = outputs.shape[1] - inputs["input_ids"].shape[1]
context_length = getattr(model.config, "max_position_embeddings", "N/A")

zephyr_result = build_result_row(
    model_name="Zephyr-7B-beta (Mistral family)",
    response_text=response,
    inference_time=end_time - start_time,
    tokens_generated=tokens_generated,
    gpu_memory_gb=gpu_memory,
    context_length=context_length,
    quantized=True
)

pd.DataFrame([zephyr_result])


,Model,Quantized (4-bit),Response Quality (sample),Inference Time (s),Tokens/sec,GPU Memory (GB),Context Length,Latency (ms/token),Cost Estimate (USD)
0,Zephyr-7B-beta (Mistral family),True,Machine learning is the field of computer scie...,4.6,8.48,4.87,32768,117.96,0.000447


## Model 3: Phi-3.5-mini-instruct

This model is used to compare its performance with the other language models in this notebook.

In [22]:
# Free Zephyr before loading Phi
free_gpu(model, tokenizer)


GPU memory cleared.


In [23]:
# Load the Phi model.

MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

print("✅ Phi Loaded Successfully!")


config.json:   0%|          | 0.00/3.45k [00:00<?, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/3.98k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

✅ Phi Loaded Successfully!


In [24]:
# Benchmark the Phi model.

prompt = "Explain Machine Learning in one sentence."

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

start_time = time.time()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False
    )

end_time = time.time()

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

inference_time = end_time - start_time
print("\nInference Time:", round(inference_time, 2), "seconds")


Machine learning is a subset of artificial intelligence that involves the use of algorithms and statistical models to enable computers to improve their performance on a specific task through experience and data, without being explicitly programmed for that task.

Inference Time: 2.2 seconds


In [25]:
# Measure GPU memory usage.

gpu_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)

print("GPU Memory Used:", round(gpu_memory, 2), "GB")


GPU Memory Used: 10.97 GB


In [26]:
# Save Phi's results row

tokens_generated = outputs.shape[1] - inputs["input_ids"].shape[1]
context_length = getattr(model.config, "max_position_embeddings", "N/A")

phi_result = build_result_row(
    model_name="Phi-3.5-mini-instruct",
    response_text=response,
    inference_time=inference_time,
    tokens_generated=tokens_generated,
    gpu_memory_gb=gpu_memory,
    context_length=context_length,
    quantized=False
)

pd.DataFrame([phi_result])


,Model,Quantized (4-bit),Response Quality (sample),Inference Time (s),Tokens/sec,GPU Memory (GB),Context Length,Latency (ms/token),Cost Estimate (USD)
0,Phi-3.5-mini-instruct,False,Machine learning is a subset of artificial int...,2.2,19.51,10.97,131072,51.25,0.000214


In [27]:
pd.DataFrame([phi_result]).to_csv("phi_benchmark.csv", index=False)

print("Phi benchmark saved successfully!")


Phi benchmark saved successfully!


In [28]:
# Free GPU memory before loading the next model

free_gpu(model, tokenizer)


GPU memory cleared.


## Model 4: DeepSeek-R1-Distill-Qwen-1.5B

In [29]:
# Load DeepSeek model

MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

print("✅ DeepSeek Loaded Successfully!")


config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.55GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

✅ DeepSeek Loaded Successfully!


In [30]:
# Benchmark DeepSeek: same prompt, same method as the other models

prompt = "Explain Machine Learning in one sentence."

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

start_time = time.time()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False
    )

end_time = time.time()

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

inference_time = end_time - start_time
print(f"\nInference Time: {inference_time:.2f} seconds")

if torch.cuda.is_available():
    gpu_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)
else:
    gpu_memory = 0

print(f"GPU Memory Used: {gpu_memory:.2f} GB")


Okay, so I need to explain machine learning in one sentence. Hmm, where do I start? I remember that machine learning is a subset of AI, but I'm not exactly sure about the details. Let me think. It's about algorithms that can learn from data, right? So maybe something like "Machine learning enables systems to learn from data and improve over time." That sounds good. But

Inference Time: 3.61 seconds
GPU Memory Used: 10.44 GB


In [31]:
# Save DeepSeek's results row

tokens_generated = outputs.shape[1] - inputs["input_ids"].shape[1]
context_length = getattr(model.config, "max_position_embeddings", "N/A")

deepseek_result = build_result_row(
    model_name="DeepSeek-R1-Distill-Qwen-1.5B",
    response_text=response,
    inference_time=inference_time,
    tokens_generated=tokens_generated,
    gpu_memory_gb=gpu_memory,
    context_length=context_length,
    quantized=False
)

pd.DataFrame([deepseek_result])


,Model,Quantized (4-bit),Response Quality (sample),Inference Time (s),Tokens/sec,GPU Memory (GB),Context Length,Latency (ms/token),Cost Estimate (USD)
0,DeepSeek-R1-Distill-Qwen-1.5B,False,"Okay, so I need to explain machine learning in...",3.61,22.18,10.44,131072,45.08,0.000351


In [32]:
pd.DataFrame([deepseek_result]).to_csv("deepseek_benchmark.csv", index=False)

print("DeepSeek benchmark saved successfully!")


DeepSeek benchmark saved successfully!


In [33]:
# Free GPU memory before loading the next model

free_gpu(model, tokenizer)


GPU memory cleared.


## Model 5: TinyLlama-1.1B-Chat

This model is used to compare its performance with the other language models in this notebook.

In [34]:
# Load TinyLlama model

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

print("✅ TinyLlama Loaded Successfully!")


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ TinyLlama Loaded Successfully!


In [35]:
# Benchmark the TinyLlama model.

prompt = "Explain Machine Learning in one sentence."

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

start_time = time.time()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False
    )

end_time = time.time()

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

inference_time = end_time - start_time
print(f"\nInference Time: {inference_time:.2f} seconds")

if torch.cuda.is_available():
    gpu_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)
else:
    gpu_memory = 0

print(f"GPU Memory Used: {gpu_memory:.2f} GB")


[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Machine Learning is a field that uses algorithms and data to improve the performance of machines and systems. It involves the development of algorithms that can learn from data and make predictions based on it.

Inference Time: 1.42 seconds
GPU Memory Used: 5.37 GB


In [36]:
# Save TinyLlama's results row

tokens_generated = outputs.shape[1] - inputs["input_ids"].shape[1]
context_length = getattr(model.config, "max_position_embeddings", "N/A")

tinyllama_result = build_result_row(
    model_name="TinyLlama-1.1B-Chat (Llama family)",
    response_text=response,
    inference_time=inference_time,
    tokens_generated=tokens_generated,
    gpu_memory_gb=gpu_memory,
    context_length=context_length,
    quantized=False
)

pd.DataFrame([tinyllama_result])


,Model,Quantized (4-bit),Response Quality (sample),Inference Time (s),Tokens/sec,GPU Memory (GB),Context Length,Latency (ms/token),Cost Estimate (USD)
0,TinyLlama-1.1B-Chat (Llama family),False,Machine Learning is a field that uses algorith...,1.42,26.75,5.37,2048,37.38,0.000138


In [37]:
pd.DataFrame([tinyllama_result]).to_csv("tinyllama_benchmark.csv", index=False)

print("TinyLlama benchmark saved successfully!")


TinyLlama benchmark saved successfully!


In [38]:
# Free GPU memory now that we're done benchmarking

free_gpu(model, tokenizer)


GPU memory cleared.


## Final LLM Benchmark Comparison

The benchmark results of all tested models are shown in the table below. The results are also saved as a CSV file for future use.

In [39]:
# Combine every model's result into one dashboard table

all_results = [qwen_result, zephyr_result, phi_result, deepseek_result, tinyllama_result]

dashboard_df = pd.DataFrame(all_results)
dashboard_df


,Model,Quantized (4-bit),Response Quality (sample),Inference Time (s),Tokens/sec,GPU Memory (GB),Context Length,Latency (ms/token),Cost Estimate (USD)
0,Qwen2.5-0.5B-Instruct,False,Machine learning is a subset of artificial int...,1.91,17.25,0.93,32768,57.98,0.000186
1,Zephyr-7B-beta (Mistral family),True,Machine learning is the field of computer scie...,4.60,8.48,4.87,32768,117.96,0.000447
2,Phi-3.5-mini-instruct,False,Machine learning is a subset of artificial int...,2.20,19.51,10.97,131072,51.25,0.000214
3,DeepSeek-R1-Distill-Qwen-1.5B,False,"Okay, so I need to explain machine learning in...",3.61,22.18,10.44,131072,45.08,0.000351
4,TinyLlama-1.1B-Chat (Llama family),False,Machine Learning is a field that uses algorith...,1.42,26.75,5.37,2048,37.38,0.000138


In [40]:
# Save the full dashboard to CSV

dashboard_df.to_csv("llm_benchmarking_dashboard.csv", index=False)

print("Dashboard saved as llm_benchmarking_dashboard.csv")


Dashboard saved as llm_benchmarking_dashboard.csv


In [41]:
# Download the combined dashboard CSV

from google.colab import files

files.download("llm_benchmarking_dashboard.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>